# HW3 — Two retrievers and a model: what RAG actually sends

**Name:** Amankeldy Abylaj
**Student ID:** _(fill in)_
**Group:** _(fill in)_
**Repository:** https://github.com/abylajamankeldy/css4007-hw3

## AI tool disclosure / Использование AI-инструментов

- **Модель в Part C:** у меня не было ключа OpenAI, поэтому каждый вызов в Part C использует **`openai/gpt-oss-120b` через API Groq, совместимый с OpenAI** (ячейка создания клиента выводит название провайдера и модели). В README указана `gpt-5.6-luna`; все три системы используют одну и ту же модель, поэтому сравнение между ними остаётся корректным, однако конкретные ответы могут отличаться от ответов `gpt-5.6-luna`.
- Письменный анализ в Parts B и C, а также ответы на Final questions основаны на результатах, которые этот notebook вывел при моём запуске.
- В notebook и репозитории нет API-ключей (`.env` исключён из Git через `.gitignore`).

Перед отправкой изменений запусти notebook **сверху вниз**. Ячейка без сохранённого вывода не засчитывается.
Инструкции для каждого раздела находятся в `README.md`.

---
# Part A — relevance by hand (1 pt)

| | Text |
|---|---|
| D1 | Large language models can generate text and answer questions. |
| D2 | Retrieval-Augmented Generation retrieves relevant documents before generating an answer. |
| D3 | Embeddings represent text as numerical vectors. |
| Query | How does RAG retrieve information? |

`k1 = 1.5`, `b = 0.75`, `idf(t) = ln((N − n(t) + 0.5) / (n(t) + 0.5))`. Выполняй tokenisation согласно разделу 3 в `README.md`.

## Key formulas / Основные формулы

### 1. BM25 — итоговая оценка документа

\[
\operatorname{BM25}(q,d)=\sum_{t\in q}\operatorname{IDF}(t)\cdot\frac{f(t,d)(k_1+1)}{f(t,d)+k_1\left(1-b+b\frac{|d|}{\operatorname{avgdl}}\right)}
\]

Здесь сумма идёт по терминам запроса; \(f(t,d)\) — сколько раз термин \(t\) встречается в документе \(d\); \(|d|\) — длина документа в токенах; \(\operatorname{avgdl}\) — средняя длина документов в корпусе. Параметр \(k_1\) управляет насыщением частоты слова, а \(b\) — поправкой на длину документа. В этом notebook используются \(k_1=1.5\) и \(b=0.75\). Чем выше BM25 score, тем выше документ в выдаче.

### 2. IDF — вес редкости термина

\[
\operatorname{IDF}(t)=\ln\left(\frac{N-n(t)+0.5}{n(t)+0.5}\right)
\]

\(N\) — общее число документов, \(n(t)\) — число документов, в которых встречается термин. Редкие слова обычно получают больший вес. В реализации `rank_bm25` отрицательные IDF обрабатываются дополнительной поправкой, поэтому отдельные оценки могут немного отличаться от ручного расчёта по базовой формуле выше.

### 3. Cosine similarity — сходство embedding-векторов

\[
\operatorname{cosine}(q,d)=\frac{q\cdot d}{\|q\|\,\|d\|}=\frac{\sum_i q_i d_i}{\sqrt{\sum_i q_i^2}\sqrt{\sum_i d_i^2}}
\]

\(q\) — вектор запроса, \(d\) — вектор документа, \(q\cdot d\) — их скалярное произведение, а \(\|q\|\) и \(\|d\|\) — длины векторов. Чем ближе cosine similarity к 1, тем ближе направления векторов. В отличие от BM25, embeddings могут находить смысловое сходство даже без точного совпадения слов.

## A1 (a) — BM25 as written

**Токены каждого документа и его длина:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large · language · models · can · generate · text · and · answer · questions | 9 |
| D2 | retrieval · augmented · generation · retrieves · relevant · documents · before · generating · an · answer | 10 |
| D3 | embeddings · represent · text · as · numerical · vectors | 6 |

**avgdl** — средняя длина документа: (9 + 10 + 6) / 3 = 25 / 3 = **8.3333**

**Query terms:** `how`, `does`, `rag`, `retrieve`, `information`  (запрос приведён к нижнему регистру; stop words не удалялись, stemming не выполнялся)

**TF, DF и IDF для каждого термина запроса** (N = 3). Формула IDF: `ln((N − n + 0.5) / (n + 0.5))`:

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | 1.9459 |
| does | 0 | 0 | 0 | 0 | 1.9459 |
| rag | 0 | 0 | 0 | 0 | 1.9459 |
| retrieve | 0 | 0 | 0 | 0 | 1.9459 |
| information | 0 | 0 | 0 | 0 | 1.9459 |

- `how`: n(t) = 0 → idf = ln((3 − 0 + 0.5) / (0 + 0.5)) = ln(3.5 / 0.5) = ln(7.0000) = **1.9459**
- `does`: n(t) = 0 → idf = ln((3 − 0 + 0.5) / (0 + 0.5)) = ln(3.5 / 0.5) = ln(7.0000) = **1.9459**
- `rag`: n(t) = 0 → idf = ln((3 − 0 + 0.5) / (0 + 0.5)) = ln(3.5 / 0.5) = ln(7.0000) = **1.9459**
- `retrieve`: n(t) = 0 → idf = ln((3 − 0 + 0.5) / (0 + 0.5)) = ln(3.5 / 0.5) = ln(7.0000) = **1.9459**
- `information`: n(t) = 0 → idf = ln((3 − 0 + 0.5) / (0 + 0.5)) = ln(3.5 / 0.5) = ln(7.0000) = **1.9459**

Ни один термин запроса не встречается ни в одном документе. Слова `retrieve`, `retrieves` и `retrieval` — разные токены, если не выполнять stemming. В D2 нет отдельного токена `rag`: там написано полное название "Retrieval-Augmented Generation". Слова `does`, `how` и `information` тоже нигде не встречаются.

**BM25 для каждого документа** (термин влияет на оценку, только если f(t, d) > 0; иначе числитель `idf · f · (k1+1)` равен 0):

> - **D1:** `how` f=0, `does` f=0, `rag` f=0, `retrieve` f=0, `information` f=0 → score = 0 + 0 + 0 + 0 + 0 = **0**
> - **D2:** `how` f=0, `does` f=0, `rag` f=0, `retrieve` f=0 (в D2 есть `retrieves` и `retrieval`, но нет `retrieve`), `information` f=0 → score = **0**
> - **D3:** все пять терминов имеют f=0 → score = **0**

| Document | BM25 Score | Rank |
|---|---:|---:|
| D1 | 0.0000 | 1 (tie) |
| D2 | 0.0000 | 1 (tie) |
| D3 | 0.0000 | 1 (tie) |

Одинаковые нулевые оценки — не арифметическая ошибка. Проверка показала, что ни один токен запроса не совпадает с токенами документов, поэтому BM25 нечего ранжировать. В этом случае BM25 не может выбрать документ на первом месте.

## A1 (b) — BM25 with the stems from README §3

Использованные основы (stems): retrieve / retrieves / retrieval → `retriev`; generate / generating / generation → `gener`; information → `inform`; does → `doe`. Все остальные слова остаются без изменений.

**Токены каждого документа после stemming и его длина:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large · language · models · can · gener · text · and · answer · questions | 9 |
| D2 | retriev · augmented · gener · retriev · relevant · documents · before · gener · an · answer | 10 |
| D3 | embeddings · represent · text · as · numerical · vectors | 6 |

**avgdl** = (9 + 10 + 6) / 3 = 25 / 3 = **8.3333** (stemming не меняет количество токенов)

**Stemmed query terms:** `how`, `doe`, `rag`, `retriev`, `inform`

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | 1.9459 |
| doe | 0 | 0 | 0 | 0 | 1.9459 |
| rag | 0 | 0 | 0 | 0 | 1.9459 |
| retriev | 0 | 2 | 0 | 1 | 0.5108 |
| inform | 0 | 0 | 0 | 0 | 1.9459 |

- `how`: n(t) = 0 → idf = ln((3 − 0 + 0.5) / (0 + 0.5)) = ln(3.5 / 0.5) = ln(7.0000) = **1.9459**
- `doe`: n(t) = 0 → idf = ln((3 − 0 + 0.5) / (0 + 0.5)) = ln(3.5 / 0.5) = ln(7.0000) = **1.9459**
- `rag`: n(t) = 0 → idf = ln((3 − 0 + 0.5) / (0 + 0.5)) = ln(3.5 / 0.5) = ln(7.0000) = **1.9459**
- `retriev`: n(t) = 1 → idf = ln((3 − 1 + 0.5) / (1 + 0.5)) = ln(2.5 / 1.5) = ln(1.6667) = **0.5108**
- `inform`: n(t) = 0 → idf = ln((3 − 0 + 0.5) / (0 + 0.5)) = ln(3.5 / 0.5) = ln(7.0000) = **1.9459**

Теперь совпадает только `retriev`, и только в D2 (два раза: `retrieval` и `retrieves`). В D1 и D3 нет терминов запроса. В D1 есть `gener`, но это не термин запроса.

**Расчёт BM25 для каждого документа:**

> **D1** = 0 (в D1 нет ни одного из терминов `how`, `doe`, `rag`, `retriev`, `inform`)
>
> **D2** — вклад даёт только термин `retriev`: 
>
> - **D2**, термин `retriev`:  idf = 0.5108, f = 2, |d| = 10, avgdl = 8.3333
>   - часть, учитывающая длину документа:  1 − b + b·|d|/avgdl = 1 − 0.75 + 0.75·10/8.3333 = 1.1500
>   - знаменатель:  f + k1·(that) = 2 + 1.5·1.1500 = 3.7250
>   - числитель:  idf·f·(k1+1) = 0.5108·2·2.5 = 2.5541
>   - оценка термина = 2.5541 / 3.7250 = **0.6857**
>
> **D3** = 0 (в D3 нет ни одного термина запроса)

| Document | BM25 Score | Rank |
|---|---:|---:|
| D1 | 0.0000 | 2 (tie) |
| D2 | 0.6857 | 1 |
| D3 | 0.0000 | 2 (tie) |

**Which document does BM25 retrieve first, in (a) and in (b)? Why?**

> - **(a):** ни один. Все три документа получают 0, поэтому у BM25 нет документа на первом месте. В запросе есть `retrieve`, а в D2 — `retrieves` и `retrieval`. Без stemming токены должны совпадать полностью. Результат выглядел ошибочным, но повторная проверка подтвердила, что это реальное поведение exact-token matching.
> - **(b):** **D2**, score 0.6857. После stemming слова `retrieves` и `retrieval` превращаются в `retriev`, поэтому термин запроса находится в D2 дважды. D1 и D3 по-прежнему получают 0.
> - Между (a) и (b) изменилась только tokenisation, а не документы или формула. Человек тоже выбрал бы D2, но BM25 нашёл его благодаря общим основам слов, а не пониманию смысла. `rag`, `how`, `doe` и `inform` по-прежнему ни с чем не совпадают: в D2 нет слов "RAG" и "information".

## A2 — cosine similarity

```
Query = [0.8, 0.2, 0.5]
D1 = [0.7, 0.1, 0.4]   D2 = [0.9, 0.3, 0.6]   D3 = [0.1, 0.8, 0.2]
```

Формула: cosine(Q, D) = (Q · D) / (‖Q‖ · ‖D‖), где Q · D = Σ qᵢ·dᵢ, а ‖v‖ = √(Σ vᵢ²).

- ‖Query‖ = √(0.8² + 0.2² + 0.5²) = √(0.64 + 0.04 + 0.25) = √0.93 = 0.9644
- ‖D1‖ = √(0.7² + 0.1² + 0.4²) = √(0.49 + 0.01 + 0.16) = √0.66 = 0.8124
- ‖D2‖ = √(0.9² + 0.3² + 0.6²) = √(0.81 + 0.09 + 0.36) = √1.26 = 1.1225
- ‖D3‖ = √(0.1² + 0.8² + 0.2²) = √(0.01 + 0.64 + 0.04) = √0.69 = 0.8307
- cos(Query, D1) = 0.78 / (0.9644 · 0.8124) = 0.78 / 0.7835 = **0.9956**
- cos(Query, D2) = 1.08 / (0.9644 · 1.1225) = 1.08 / 1.0825 = **0.9977**
- cos(Query, D3) = 0.34 / (0.9644 · 0.8307) = 0.34 / 0.8011 = **0.4244**

| Pair | Dot product (worked) | ‖Query‖ | ‖D‖ | Cosine |
|---|---|---:|---:|---:|
| Query · D1 | 0.8·0.7 + 0.2·0.1 + 0.5·0.4 = 0.56 + 0.02 + 0.20 = 0.78 | 0.9644 | 0.8124 | 0.9956 |
| Query · D2 | 0.8·0.9 + 0.2·0.3 + 0.5·0.6 = 0.72 + 0.06 + 0.30 = 1.08 | 0.9644 | 1.1225 | 0.9977 |
| Query · D3 | 0.8·0.1 + 0.2·0.8 + 0.5·0.2 = 0.08 + 0.16 + 0.10 = 0.34 | 0.9644 | 0.8307 | 0.4244 |

| Document | Cosine Similarity | Rank |
|---|---:|---:|
| D1 | 0.9956 | 2 |
| D2 | 0.9977 | 1 |
| D3 | 0.4244 | 3 |

**Which document is most similar to the query?**

> **D2** (cosine 0.9977). D1 почти не отстаёт (0.9956), а D3 значительно дальше (0.4244): вектор D3 направлен преимущественно вдоль второй оси, тогда как вектор запроса — вдоль первой.

**Is the ranking the same as BM25's?** Сравни результат с обоими вариантами A1.

> Ранжирование по cosine similarity: **D2 > D1 > D3**.
> - С **A1 (a)** напрямую сравнить нельзя: BM25 дал всем документам 0, поэтому ранжирования фактически нет; cosine similarity выстраивает все три документа.
> - В **A1 (b)** первое место одинаковое — **D2**. Остальное отличается: BM25 оставляет D1 и D3 с одинаковой оценкой 0, а cosine ставит D1 на второе место (0.9956), D3 — на последнее (0.4244). После stemming оба метода выбирают D2, а cosine дополнительно показывает близость D1.

**Why can BM25 and embedding retrieval disagree?** Point at the words of this query and these documents.

> BM25 считает точные совпадения токенов. Слова `retrieve` и `information` в исходном виде не встречаются ни в одном документе, поэтому в (a) у BM25 нет совпадений. Даже в (b) он находит только `retriev` в D2 и не распознаёт, что D1 ("answer questions") тоже относится к теме. Embedding-модель сравнивает смысл: запрос "How does RAG retrieve information?" близок к "Retrieval-Augmented Generation retrieves relevant documents…", потому что RAG — это retrieval-augmented generation; BM25 не может вывести это значение из букв `rag`. Запрос также близок к D1 о языковых моделях, отвечающих на вопросы, хотя общих слов нет. Обратная слабость embeddings — они могут хуже различать точные строки (например, код формы GR-305), где BM25 силён.
> Здесь векторы заданы в условии, поэтому смысл уже отражён в числах. D1 и D2 близки к запросу, потому что их векторы направлены почти одинаково.

Проверка вычислений в Part A (все расчёты выше выполнены вручную; эта ячейка лишь подтверждает численные результаты):

In [1]:
# Вручную реализуем BM25 и косинусное сходство, чтобы проверить формулы на небольшом примере.
import math
# k1 управляет насыщением частоты слова, b — силой поправки на длину документа.
k1, b = 1.5, 0.75
# Три коротких документа из условия; split() превращает текст в список слов.
D = {"D1": "large language models can generate text and answer questions".split(),
     "D2": "retrieval augmented generation retrieves relevant documents before generating an answer".split(),
     "D3": "embeddings represent text as numerical vectors".split()}
# Правила стемминга сводят несколько форм одного слова к общей основе.
STEM = {"retrieve": "retriev", "retrieves": "retriev", "retrieval": "retriev", "generate": "gener",
        "generating": "gener", "generation": "gener", "information": "inform", "does": "doe"}
# Разбиваем поисковый запрос на слова.
query = "how does rag retrieve information".split()

# Считаем BM25 для каждого документа и возвращаем оценки по ID.
def bm25_by_hand(docs, q):
# N — количество документов; avg — средняя длина документа в словах.
    N = len(docs); avg = sum(len(w) for w in docs.values()) / N
    scores = {}
    for d, w in docs.items():
        s = 0
        for t in q:
# n — число документов, в которых встречается термин; это нужно для IDF.
            n = sum(1 for ww in docs.values() if t in ww)
# IDF повышает вес редких терминов и снижает вес распространённых.
            idf = math.log((N - n + 0.5) / (n + 0.5))
# f — частота термина в текущем документе.
            f = w.count(t)
# Добавляем вклад термина с поправками на частоту и длину документа.
            s += idf * f * (k1 + 1) / (f + k1 * (1 - b + b * len(w) / avg))
        scores[d] = round(s, 4)
    return scores

print("(a) as written :", bm25_by_hand(D, query))
# Применяем стемминг к словам документов.
Ds = {d: [STEM.get(x, x) for x in w] for d, w in D.items()}
print("(b) with stems :", bm25_by_hand(Ds, [STEM.get(x, x) for x in query]))

# Вектор запроса и векторы документов заданы в условии.
Q = [0.8, 0.2, 0.5]
V = {"D1": [0.7, 0.1, 0.4], "D2": [0.9, 0.3, 0.6], "D3": [0.1, 0.8, 0.2]}
# Норма вектора нужна в знаменателе косинусного сходства.
norm = lambda v: math.sqrt(sum(x * x for x in v))
# Для каждого документа считаем скалярное произведение и cosine similarity.
for d, v in V.items():
    dot = sum(x * y for x, y in zip(Q, v))
    print(d, "dot =", round(dot, 2), "|Q| =", round(norm(Q), 4), "|D| =", round(norm(v), 4), "cos =", round(dot / (norm(Q) * norm(v)), 4))

(a) as written : {'D1': 0.0, 'D2': 0.0, 'D3': 0.0}
(b) with stems : {'D1': 0.0, 'D2': 0.6857, 'D3': 0.0}
D1 dot = 0.78 |Q| = 0.9644 |D| = 0.8124 cos = 0.9956
D2 dot = 1.08 |Q| = 0.9644 |D| = 1.1225 cos = 0.9977
D3 dot = 0.34 |Q| = 0.9644 |D| = 0.8307 cos = 0.4244


---
# Part B — the same two retrievers, from libraries (1.5 pt)

Загрузи `data/corpus.jsonl` и `data/queries.json`.

In [2]:
# Библиотеки для чтения файлов, регулярных выражений, переменных окружения и математики.
import json, re, os, math
import numpy as np
from dotenv import load_dotenv
# Загружаем переменные из локального .env; секретные ключи не выводятся на экран.
load_dotenv()          # reads OPENAI_API_KEY / GROQ_API_KEY / HF_TOKEN from .env (never printed)

# Читаем корпус фрагментов, запросы и контрольные вопросы из файлов data/.
corpus  = [json.loads(l) for l in open("data/corpus.jsonl", encoding="utf-8")]
queries = json.load(open("data/queries.json", encoding="utf-8"))["queries"]
qa      = json.load(open("data/qa.json", encoding="utf-8"))["questions"]
# Создаём список ID и словарь для быстрого доступа к фрагменту по ID.
ids     = [c["id"] for c in corpus]
by_id   = {c["id"]: c for c in corpus}
print(len(corpus), "chunks,", len(queries), "fixed queries,", len(qa), "questions")

74 chunks, 8 fixed queries, 6 questions


## B1 — BM25 with `rank_bm25`

**Моя tokenisation:** сначала нижний регистр, затем `re.findall(r"\w+", text)` (токен — последовательность букв или цифр; регулярное выражение поддерживает Unicode, поэтому русские и казахские слова остаются целыми). **Stemming и удаление stop words не выполняются**, поэтому `retrieve` и `retrieves` остаются разными токенами, как в A1 (a). Код формы `GR-305` разбивается на два токена — `gr` и `305`, потому что дефис служит разделителем. Используются параметры `BM25Okapi(k1=1.5, b=0.75)`.

In [3]:
# Используем готовую реализацию BM25 из библиотеки rank_bm25.
from rank_bm25 import BM25Okapi

# Токенизация: нижний регистр и последовательности букв/цифр; стоп-слова не удаляются.
def tok(s):
    return re.findall(r"\w+", s.lower())

# Создаём индекс BM25 для текстов корпуса; k1 и b задают параметры ранжирования.
bm25 = BM25Okapi([tok(c["text"]) for c in corpus], k1=1.5, b=0.75)

# Ищем k лучших фрагментов и возвращаем их ID вместе с оценками.
def search_bm25(q, k=3):
# Считаем BM25-оценку каждого документа для запроса.
    s = bm25.get_scores(tok(q))
# Сортируем оценки по убыванию; stable сохраняет порядок при равных оценках.
    top = np.argsort(-s, kind="stable")[:k]
    return [(ids[i], float(s[i])) for i in top]

# Формируем короткую подпись фрагмента, чтобы результаты было легче читать.
def first_line(cid, n=80):
    c = by_id[cid]
    return (c["title"] + " | " + c["text"].split("\n")[0])[:n]

print("example:", search_bm25(queries[0]["text"]))

example: [('GO-08', 15.825491699499644), ('GO-12', 8.195256931904641), ('GO-06', 8.07003113015204)]


## B2 — EmbeddingGemma (`google/embeddinggemma-300m`) с методами `encode_document` / `encode_query`

In [4]:
# Импортируем класс для загрузки embedding-модели.
from sentence_transformers import SentenceTransformer
import os
import numpy as np

# Загружаем EmbeddingGemma; HF_TOKEN берётся из переменных окружения, если нужен.
model = SentenceTransformer(
    "google/embeddinggemma-300m",
    token=os.environ.get("HF_TOKEN")
)

# Кодируем тексты корпуса в числовые векторы (embeddings) и храним их в памяти.
doc_emb = model.encode_document(
    [c["text"] for c in corpus],
    show_progress_bar=False
)

print("document embeddings:", doc_emb.shape, doc_emb.dtype)

# Ищем наиболее похожие на запрос фрагменты по embedding-векторам.
def search_emb(q, k=3):
# Кодируем текст именно как поисковый запрос.
    qe = model.encode_query(q)
# Считаем сходство запроса со всеми векторами документов.
    s = np.asarray(model.similarity(qe, doc_emb)[0])
    top = np.argsort(-s, kind="stable")[:k]
    return [(ids[i], float(s[i])) for i in top]

print("example:", search_emb(queries[0]["text"]))

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

document embeddings: (74, 768) float32
example: [('GO-08', 0.6955164074897766), ('GO-12', 0.4608723521232605), ('GO-23', 0.45048606395721436)]


## My own queries

Эти запросы составлены **до** запуска поисковых методов. Значения `relevant` определены по handbook, а не по результатам поиска.

| Id | Query | `relevant` | Expected winner, and why |
|---|---|---|---|
| Q-09 | Which office is in room 214? | GO-21 | **BM25.** Запрос опирается на точный токен `214` (и `room`), который встречается только в одном chunk. Embedding-модель может хуже различать числа, а слово "office" способно подтянуть любой chunk о Grant Office. |
| Q-10 | Какие документы подтверждают доход семьи? | GO-10 | **Embeddings.** Вопрос задан на русском языке ("какие документы подтверждают доход семьи"), а chunk GO-10 с ответом ("Proof of income … salary certificate …") написан по-английски. У BM25 нет общих слов между этими языками, но multilingual embedding-модель должна сопоставить их по смыслу. |

In [5]:
# Два дополнительных запроса для проверки, включая вопрос на русском языке.
own = [
    {"id": "Q-09", "text": "Which office is in room 214?",                "relevant": ["GO-21"]},
    {"id": "Q-10", "text": "Какие документы подтверждают доход семьи?",    "relevant": ["GO-10"]},
]
# Объединяем исходные и собственные запросы для единого сравнения.
allq = queries + own
print(len(allq), "queries in total")

10 queries in total


## Results for all ten or more queries: top 3 от каждого retriever

In [6]:
# Hit@3 — True, если среди трёх найденных фрагментов есть хотя бы один правильный.
def hit(res, relevant):
    return any(i in relevant for i, _ in res)

# Сюда записываем выдачу обоих поисковиков и результат проверки Hit@3.
results = {}
# Проверяем каждый запрос обоими методами.
for q in allq:
# Получаем top-3 для BM25 и EmbeddingGemma и проверяем, попал ли правильный фрагмент.
    b, e = search_bm25(q["text"]), search_emb(q["text"])
    results[q["id"]] = {"bm25": b, "emb": e, "hb": hit(b, q["relevant"]), "he": hit(e, q["relevant"])}
    print(f"{q['id']}  {q['text']}")
    print(f"   relevant: {q['relevant']}")
    for name, res, h in (("BM25", b, results[q['id']]['hb']), ("EmbeddingGemma", e, results[q['id']]['he'])):
        print(f"   {name}   hit@3 = {'YES' if h else 'no'}")
        for r, (cid, sc) in enumerate(res, 1):
            mark = "*" if cid in q["relevant"] else " "
            print(f"     {r}. {cid}{mark} {sc:8.4f}  {first_line(cid)}")
    print()
print("(* = chunk is in `relevant`)")

Q-01  What is form GR-305 used for?
   relevant: ['GO-08', 'GO-12', 'GO-23']
   BM25   hit@3 = YES
     1. GO-08*  15.8255  Form GR-305: appeals | Form GR-305 is used to appeal a decision of the Grant Off
     2. GO-12*   8.1953  How to appeal a decision | A student who disagrees with a decision may appeal wi
     3. GO-06    8.0700  Form GR-104: the application form | Form GR-104 is the application form for the 
   EmbeddingGemma   hit@3 = YES
     1. GO-08*   0.6955  Form GR-305: appeals | Form GR-305 is used to appeal a decision of the Grant Off
     2. GO-12*   0.4609  How to appeal a decision | A student who disagrees with a decision may appeal wi
     3. GO-23*   0.4505  Апелляция: кратко | Если студент не согласен с решением, он может подать апелляц

Q-02  My family is poor. Is there financial help for students like me?
   relevant: ['GO-01', 'GO-02', 'GO-03']
   BM25   hit@3 = no
     1. KZ-17    8.7262  Lake Balkhash (part 2) | The lake currently covers about 16,400 km2 (6,300

## B3 — comparison

Таблица ниже автоматически выводится следующей ячейкой из того же словаря `results`, что и списки top-3 выше. После таблицы приведены три примера.

In [7]:
# Импортируем средства Jupyter для отображения таблицы в Markdown.
from IPython.display import Markdown, display
# Создаём заголовок таблицы сравнения поисковых методов.
rows = ["| Query | BM25 top 3 (ids) | BM25 hit@3 | Embedding top 3 (ids) | Embedding hit@3 |", "|---|---|:---:|---|:---:|"]
# Добавляем строку с результатами для каждого запроса.
for q in allq:
    r = results[q["id"]]
    rows.append(f"| {q['id']} | {', '.join(i for i, _ in r['bm25'])} | {'✓' if r['hb'] else '✗'} | {', '.join(i for i, _ in r['emb'])} | {'✓' if r['he'] else '✗'} |")
# Считаем, для скольких запросов каждый метод нашёл правильный фрагмент в top-3.
tb = sum(r["hb"] for r in results.values()); te = sum(r["he"] for r in results.values())
rows.append(f"| **hit@3 total** | | **{tb}/{len(allq)}** | | **{te}/{len(allq)}** |")
display(Markdown("\n".join(rows)))

| Query | BM25 top 3 (ids) | BM25 hit@3 | Embedding top 3 (ids) | Embedding hit@3 |
|---|---|:---:|---|:---:|
| Q-01 | GO-08, GO-12, GO-06 | ✓ | GO-08, GO-12, GO-23 | ✓ |
| Q-02 | KZ-17, GO-20, KZ-25 | ✗ | GO-01, GO-10, GO-20 | ✓ |
| Q-03 | GO-14, KZ-01, KZ-02 | ✓ | GO-14, GO-09, GO-11 | ✓ |
| Q-04 | GO-05, GO-17, GO-19 | ✗ | GO-02, GO-05, GO-17 | ✓ |
| Q-05 | KZ-05, KZ-06, KZ-11 | ✓ | KZ-05, KZ-06, KZ-03 | ✓ |
| Q-06 | KZ-37, KZ-38, KZ-03 | ✓ | KZ-37, KZ-38, KZ-10 | ✓ |
| Q-07 | GO-21, GO-16, GO-01 | ✗ | GO-15, GO-18, GO-01 | ✓ |
| Q-08 | GO-17, GO-05, GO-04 | ✓ | GO-17, GO-18, GO-15 | ✓ |
| Q-09 | GO-21, GO-08, GO-07 | ✓ | GO-21, GO-07, KZ-37 | ✓ |
| Q-10 | KZ-01, KZ-02, KZ-03 | ✗ | GO-10, GO-07, GO-13 | ✓ |
| **hit@3 total** | | **6/10** | | **10/10** |

**Оба retriever согласны:  Q-06** ("Where did the Soviet Union test its nuclear weapons?").
Оба метода вернули два одинаковых релевантных chunk в одном порядке: сначала KZ-37 (BM25 19.09, EmbeddingGemma 0.5960), затем KZ-38 (BM25 17.17, EmbeddingGemma 0.5102). Этот запрос оказался простым, потому что слова вопроса совпадают со словами ответа: Токены `soviet`, `union`, `test`, `nuclear` встречаются в обоих chunk (KZ-38 начинается с фразы "The Soviet Union conducted 456 nuclear tests at Semipalatinsk"). Смысл тоже совпадает, поэтому поиск по точным словам и по смыслу приводит к одному результату. Различается только третье место (BM25: KZ-03, embeddings: KZ-10), и ни один из этих chunk не относится к ответу. Для Q-05 (Astana) результат похож: оба метода ставят KZ-05 на первое место (15.73 и 0.6514).

**BM25 did better: честный ответ — ни одного запроса, если под «лучше» понимать hit@3.** BM25 получил 6/10, а EmbeddingGemma — 10/10; нет ни одного запроса, который BM25 обработал успешно, а embeddings пропустили. Точный поиск по строкам явно помог в запросе **Q-09** ("Which office is in room 214?"): оба метода нашли GO-21, но BM25 совпал с точными токенами `214` и `room`, каждый из которых встречается только в одном из 74 chunk (каждый добавил 4.39 к оценке GO-21, равной 14.73). Поэтому GO-21 опередил следующий результат GO-08 с оценкой 6.94 более чем в два раза. Embedding-модель тоже поставила GO-21 на первое место, но разрыв был меньше (0.4533 против 0.2483), а на третье место попал нерелевантный chunk о Семипалатинске (KZ-37): embeddings не всегда точно различают отдельные числа вроде `214`. Та же точность иногда мешала BM25. В **Q-04** BM25 совпал со словами `minimum` (4.57) и `gpa` (2.98) в архивном chunk GO-05 ("the minimum GPA for the grant was 3.0") и поставил его на первое место (12.78). Правильный chunk GO-02 содержит формулировку "at least 2.67", но не содержит слова `minimum`, поэтому выпал из top 3. Заголовок chunk, который оказался первым: "Archived: the 2025 scheme (no longer in force)": BM25 не понимает, что этот заголовок означает «не используй этот фрагмент».

**Embeddings did better: Q-07** ("Can the grant be sent to my mother's card?"). Ответ находится в GO-15 на русском языке: "перевод на счёт родителей или других лиц невозможен" (transfer to the account of parents or other persons is impossible). У BM25 **нет ни одного общего токена** с GO-15 (вклад в оценку равен нулю), поэтому он не нашёл этот фрагмент. Его top 3 (GO-21, GO-16, GO-01) появились из-за слов `sent`, `can`, `grant`. BM25 не смог сопоставить разные языки (английский запрос и русский chunk) или перефразирование "mother's card" ↔ "счёт родителей". EmbeddingGemma поставила GO-15 на первое место (0.4216), но лишь немного выше GO-18 (0.4171). Та же закономерность объясняет другие победы embeddings: **Q-10** (вопрос на русском, английский chunk с ответом GO-10: BM25 получил 0.0000 для всех документов) и **Q-02** ("My family is poor" против формулировки handbook "low-income household": BM25 совпал только со словами `like`, `there`, `is` и первым вернул chunk о Lake Balkhash).

---
# Part C — connect retrieval to the model (1.5 pt)

Модель: `gpt-5.6-luna` для каждого вызова. Используется top 3, если не указано иное.

In [8]:

# Для Part C нужны API-клиент, задержки между запросами и подсчёт токенов.
import os
import time
import tiktoken
from dotenv import load_dotenv
from openai import OpenAI

# Перечитываем .env, позволяя его значениям заменить ранее заданные переменные окружения.
load_dotenv(override=True)

# Выбираем кодировщик для локальной оценки количества токенов.
enc = tiktoken.get_encoding("o200k_base")

# Возвращает число токенов в строке по выбранному кодировщику.
def ntok(s):
    return len(enc.encode(s))

# Если задан ключ Groq, используем его API с совместимым с OpenAI интерфейсом.
if os.getenv("GROQ_API_KEY"):
    client = OpenAI(
        api_key=os.environ["GROQ_API_KEY"],
        base_url="https://api.groq.com/openai/v1"
    )
    MODEL = "openai/gpt-oss-120b"
    GROQ = True
# Если ключ Groq отсутствует, проверяем наличие ключа OpenAI.
elif os.getenv("OPENAI_API_KEY"):
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    MODEL = "gpt-5.6-luna"
    GROQ = False
else:
    raise SystemExit("Укажи GROQ_API_KEY или OPENAI_API_KEY в .env")

print(
    "provider:", "Groq" if GROQ else "OpenAI",
    "| model:", MODEL
)

# Запоминаем время прошлого вызова, чтобы ограничивать частоту запросов.
_last = [0.0]

# Отправляет сообщения модели и возвращает текст ответа и статистику токенов.
def ask(messages):
    """one chat call -> (answer, prompt_tokens, completion_tokens)"""
    kw = {"max_tokens": 4000, "temperature": 0} if GROQ else {}

# Повторяем запрос при временной ошибке, максимум шесть попыток.
    for attempt in range(6):
        wait = 2.5 - (time.time() - _last[0])
        if GROQ and wait > 0:
            time.sleep(wait)

        _last[0] = time.time()

# Если вызов API не удался, обработчик ниже сделает паузу и повторит попытку.
        try:
            r = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                **kw
            )
            return (
                (r.choices[0].message.content or "").strip(),
                r.usage.prompt_tokens,
                r.usage.completion_tokens
            )
        except Exception as ex:
            if attempt == 5:
                raise
            print("Retry after error:", type(ex).__name__)
            time.sleep(5 * (attempt + 1))

# Системная инструкция запрещает опираться на сведения вне контекста и просит цитировать блоки.
SYSTEM = """Answer the question using only the provided context.
If the answer cannot be found in the context, say that you do not know.
Cite the context blocks you used, like [1] or [2][3]."""

# Преобразуем ID найденных фрагментов в пронумерованный текстовый контекст.
def context_block(chunk_ids):
    return "\n\n".join(
        f"[{n}] ({cid} | {by_id[cid]['title']})\n{by_id[cid]['text']}"
        for n, cid in enumerate(chunk_ids, 1)
    )

# Собираем сообщения system и user для запроса в формате RAG.
def rag_messages(question, chunk_ids):
    user = (
        "Context:\n" + context_block(chunk_ids)
        + "\n\nQuestion: " + question
    )
    return [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": user}
    ]


provider: Groq | model: openai/gpt-oss-120b


## C1 — the complete prompt, printed before it is sent

In [9]:
# Выбираем вопрос C-02 и смотрим top-3, найденный embedding-поиском.
q1 = next(x for x in qa if x["id"] == "C-02")
top = search_emb(q1["question"], 3)
c1_ids = [cid for cid, _ in top]
print("EmbeddingGemma top 3:", top, "\n")

# Формируем полный prompt и печатаем его до отправки модели.
msgs = rag_messages(q1["question"], c1_ids)
print("=" * 30, "WHAT THE MODEL RECEIVES", "=" * 30)
for m in msgs:
    print(f"--- role: {m['role']} ---")
    print(m["content"])
    print()
print("=" * 85)

# Считаем токены отдельно для инструкции, вопроса и найденного контекста.
# token accounting with tiktoken o200k_base
# Оцениваем размер каждой части prompt в токенах.
sys_t  = ntok(SYSTEM)
q_t    = ntok("Question: " + q1["question"])
ctx_text = "Context:\n" + context_block(c1_ids)
ctx_t  = ntok(ctx_text)
total_parts = sys_t + q_t + ctx_t
whole = ntok(msgs[0]["content"]) + ntok(msgs[1]["content"])

# Отправляем prompt модели и получаем фактическую статистику токенов.
answer, prompt_tokens, completion_tokens = ask(msgs)
print("answer:", answer, "\n")

from IPython.display import Markdown, display
display(Markdown(f"""| Part of the input | Tokens (tiktoken `o200k_base`) |
|---|---:|
| System instructions | {sys_t} |
| Question (with the label "Question:") | {q_t} |
| Retrieved context (with the label "Context:" and the block headers) | {ctx_t} |
| **Total input** (sum of the three) | **{total_parts}** |
| **`usage.prompt_tokens` from the response** | **{prompt_tokens}** |"""))
print("tiktoken on the two message strings as a whole:", whole)
print("difference usage - tiktoken sum:", prompt_tokens - total_parts)
# Считаем размер каждого найденного фрагмента, чтобы оценить объём контекста.
per_chunk = [ntok(by_id[c]["text"]) for c in c1_ids]
print("number of retrieved chunks:", len(c1_ids))
print("tokens per chunk (text only):", dict(zip(c1_ids, per_chunk)))
print("total context size:", ctx_t, "tokens,", len(ctx_text), "characters  (chunk text only:", sum(per_chunk), "tokens,", sum(len(by_id[c]["text"]) for c in c1_ids), "characters)")

EmbeddingGemma top 3: [('GO-08', 0.6028919219970703), ('GO-12', 0.5590736269950867), ('GO-11', 0.5342729091644287)] 

============================== WHAT THE MODEL RECEIVES ==============================
--- role: system ---
Answer the question using only the provided context.
If the answer cannot be found in the context, say that you do not know.
Cite the context blocks you used, like [1] or [2][3].

--- role: user ---
Context:
[1] (GO-08 | Form GR-305: appeals)
Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it cannot be used to apply for the first time or to report a change in circumstances. On GR-305 the student states which part of the decision they believe is wrong and which rule in this handbook supports their case.

[2] (GO-12 | How to appeal a decision)
A student who disagrees with a decision may appeal within 14 calendar days of the date on the decision letter, using form GR-305. The appeal is read by a panel of three staf

| Part of the input | Tokens (tiktoken `o200k_base`) |
|---|---:|
| System instructions | 44 |
| Question (with the label "Question:") | 23 |
| Retrieved context (with the label "Context:" and the block headers) | 279 |
| **Total input** (sum of the three) | **346** |
| **`usage.prompt_tokens` from the response** | **420** |

tiktoken on the two message strings as a whole: 346
difference usage - tiktoken sum: 74
number of retrieved chunks: 3
tokens per chunk (text only): {'GO-08': 68, 'GO-12': 88, 'GO-11': 78}
total context size: 279 tokens, 1278 characters  (chunk text only: 234 tokens, 1150 characters)


Таблица выше автоматически выводится предыдущей ячейкой на основе `ntok(...)` и `response.usage`; она не введена вручную.

Использованный вопрос: **C-02**. EmbeddingGemma retrieved GO-08 (0.6029), GO-12 (0.5591), GO-11 (0.5343). The model answered "You have **14 calendar days** from the date on the decision letter ... **form GR‑305**. [2]", which matches `reference`.

Результаты моего запуска: system 44 + question 23 + context 279 = **346** tokens by tiktoken; the response reported `usage.prompt_tokens` = **420**; the gap is **74** tokens. 3 chunks of 68, 88 and 78 tokens (234 tokens, 1150 characters of chunk text); with the "Context:" label and the `[n] (id | title)` headers the context block is 279 tokens and 1278 characters. The context is 279 of the 346 counted tokens (about 81%).

**Why do the tiktoken total and `usage.prompt_tokens` differ?**

> `tiktoken` считает только написанный мной текст, используя словарь `o200k_base`. Сервер считает то, что модель действительно получает: два моих сообщения **обёрнуты в chat template**, то есть добавляются специальные токены начала и конца каждого сообщения и обозначения его роли. Для этой модели сервер также добавляет вокруг system message дополнительный текст (стандартный preamble). Я не вижу эту обёртку напрямую, но мои запуски позволяют оценить её размер: в C-02 вызов "LLM only" отправлял только вопрос из одного предложения (около 20 токенов), а `usage.prompt_tokens` показывал **92**; в C-03 вопрос примерно из 17 токенов дал **85**. Значит, к каждому запросу добавляется около 70 токенов — это объясняет разницу в 74 токена здесь. Кроме того, `o200k_base` — словарь OpenAI, а модель обслуживается через Groq, поэтому подсчёт самого текста тоже может немного отличаться. Такая же разница была в HW1: количество токенов, которое я считаю локально, не обязательно равно количеству, которое учитывается сервером.

**What exactly does retrieval add to the model's input?**

> На вход модели передаётся только **текст**. Три chunk, дословно скопированные из `corpus.jsonl`, каждый с заголовком вроде `[n] (GO-08 | Form GR-305: appeals)`, а также слово `Context:`. В этом prompt контекст занимает 279 токенов и 1278 символов: в 12 раз больше вопроса (23 токена) и в 6 раз больше system message (44 токена). Важное сравнение — с вызовом без retrieval: тот же вопрос занимал 92 токена в режиме `LLM only` и 420 токенов с retrieval, то есть retrieval вместе с инструкциями добавил **328 токенов**. Модель не получает документ целиком, индекс, векторы или similarity scores. Она не может самостоятельно открыть GO-08: если chunk не попал в top 3, для модели его как будто нет. Всё, что модель «знает» из handbook в этом ответе, содержится в этих 234 токенах текста chunk.

## C2 — three системы, шесть вопросов

In [10]:
# Сохраняем ответы трёх вариантов системы для каждого контрольного вопроса.
runs = {}
# Для каждого вопроса сравниваем LLM без контекста и два варианта RAG.
for q in qa:
    qid = q["id"]
# Получаем ID трёх фрагментов, найденных BM25 и EmbeddingGemma.
    b_ids = [cid for cid, _ in search_bm25(q["question"], 3)]
    e_ids = [cid for cid, _ in search_emb(q["question"], 3)]
# Вариант 1: модель получает только вопрос, без найденных документов.
    a_llm, pt_llm, _ = ask([{"role": "user", "content": q["question"]}])            # system 1: no context
# Вариант 2: модель получает вопрос и контекст, найденный BM25.
    a_bm,  pt_bm,  _ = ask(rag_messages(q["question"], b_ids))                        # system 2: BM25 top 3
# Вариант 3: модель получает вопрос и контекст, найденный EmbeddingGemma.
    a_em,  pt_em,  _ = ask(rag_messages(q["question"], e_ids))                        # system 3: embedding top 3
# Сохраняем ответы, ID найденных фрагментов и число prompt-токенов для сравнения.
    runs[qid] = {"llm": a_llm, "bm25": a_bm, "emb": a_em, "bm25_ids": b_ids, "emb_ids": e_ids,
                 "pt": {"llm": pt_llm, "bm25": pt_bm, "emb": pt_em}}
    print("=" * 100)
    print(qid, "|", q["question"])
    print("REFERENCE:", q["reference"])
    print("relevant chunks:", q["relevant"])
    print(f"\n[LLM only]   (prompt_tokens {pt_llm})\n{a_llm}")
    print(f"\n[BM25 RAG]   chunks {b_ids}  (prompt_tokens {pt_bm})\n{a_bm}")
    print(f"\n[Embedding RAG] chunks {e_ids}  (prompt_tokens {pt_em})\n{a_em}")

C-01 | How much does a student in income band 2 receive, and when is it paid?
REFERENCE: 150,000 tenge for the academic year, in two equal instalments: October and February.
relevant chunks: ['GO-04', 'GO-15']

[LLM only]   (prompt_tokens 89)
**What you’ll get**

In England the “income‑band 2” category is for households whose **adjusted net income is between £25,001 and £30,000** (the exact figure you report on the application determines where you fall inside the band).

| Household income | Maintenance‑loan amount (2023‑24 rates) |
|------------------|------------------------------------------|
| £25,001 – £26,000 | £9 500 |
| £26,001 – £27,000 | £9 250 |
| £27,001 – £28,000 | £9 000 |
| £28,001 – £29,000 | £8 750 |
| £29,001 – £30,000 | £8 500 |

So, a student whose household income puts them in **income‑band 2** will receive **between £8 500 and £9 500** for the academic year, depending on the exact income figure you give.

**When it’s paid**

The loan is paid **directly to your ban

Ответы выше — это фактические результаты эксперимента; каждый сравнивается с `reference`. «✓» означает, что ответ совпадает с эталоном по смыслу (формулировка может отличаться), а «✗» — что не совпадает.

| Question | LLM only | BM25 RAG | Embedding RAG |
|---|---|---|---|
| C-01 | ✗ invents an England maintenance loan: £8,500–£9,500, three instalments | ✓ 150,000 tenge, two instalments, October and February [1] (chunks GO-04, GO-02, GO-01) | ✓ 150,000 tenge, two equal instalments, October and February [1][3] (GO-04, GO-03, GO-15) |
| C-02 | ✗ a long generic answer ("30 days", "NIH, NSF", "consult a lawyer"), no GR-305 | ✓ 14 calendar days, form GR-305 (GO-05, GO-08, GO-12) | ✓ 14 calendar days, form GR-305 (GO-08, GO-12, GO-11) |
| C-03 | ✗ "2.5" (invented) | ✓ 2.67 on the 4.0 scale (GO-16, GO-02, GO-06) | ✓ 2.67 on the 4.0 scale (GO-02, GO-20, GO-05) |
| C-04 | ✗ decree of 12 October 2017 "№ 1195" (number and day not in the corpus), but says the switch is due by **end of 2025**, not 2031 | ✓ October 2017; deadline moved to 2031 (KZ-43, KZ-14, KZ-06) | ✓ October 2017; target now 2031 (KZ-14, KZ-43, KZ-05) |
| C-05 | ✗ "1992, Kazakh State Institute of Economics and Business" | ✓ 1963, Almaty Institute of National Economy (KZ-44, KZ-11, KZ-07) | ✓ 1963, Almaty Institute of National Economy (KZ-44, KZ-05, KZ-14) |
| C-06 | ✗ a long US guide (FAFSA, assistantships, "$2,000–$20,000"); never says it does not know | ✓ the grant is for undergraduates; master's students have a different scheme "that isn't detailed here", amount not specified [3] (GO-16, GO-01, GO-20) | ✓ same: separate scheme not detailed, amount not specified [1] (GO-20, GO-16, GO-01) |
| **agrees with `reference`** | **0/6** | **6/6** | **6/6** |

Примечания. (1) Обе RAG-системы набрали 6/6 на этих шести вопросах, хотя в Part B BM25 нашёл правильный chunk только для 6/10 запросов. Разница здесь не видна, потому что формулировки этих шести вопросов близки к тексту chunk, а K = 3 дало BM25 дополнительный шанс (для C-01 ответ целиком содержится в GO-04). (2) В некоторых ответах появились ссылки вида 【1†L1-L4】 вместо требуемого формата `[1]`. По содержанию ответы верны, а формат цитирования — привычка самой модели.

## C3 — question 1 of 3: C-03 (minimum GPA, with an archived rule in the corpus)

**1. Что нашёл BM25** (top 3, ID и оценки):

> GO-16 (15.799), GO-02 (14.059), GO-06 (13.852).

**2. Что нашла EmbeddingGemma** (top 3, ID и оценки):

> GO-02 (0.629), GO-20 (0.569), GO-05 (0.545).

**3. Что было передано модели** (блоки контекста):

> BM25: [1] GO-16 "Combining the grant with other support" (the 400,000 tenge cap), [2] GO-02 "Who qualifies" ("cumulative GPA is at least 2.67 on the 4.0 scale"), [3] GO-06 (form GR-104). EmbeddingGemma: [1] GO-02, [2] GO-20 "Who cannot apply", [3] **GO-05 "Archived: the 2025 scheme (no longer in force)"**, whose text says "the minimum GPA for the grant was 3.0". The full blocks are printed in the cell below.

**4. Как retrieval повлиял на ответ:** сделал его точнее, конкретнее и лучше обоснованным или привёл к ошибке из-за нерелевантных chunk?

> **Ответ стал точнее и лучше обоснованным.** Без контекста модель назвала "minimum cumulative GPA of **2.5**", хотя такого числа в handbook нет. С контекстом обе RAG-системы ответили **2.67 on the 4.0 scale** и сослались на блок, в котором указано это значение ([2] для BM25, [1] для embeddings).

**5. Что было упущено и в чём причина — retrieval или generation?** Приведи подтверждение.

> В итоговых ответах ничего не упущено: оба ответа правильные. Но retrieval сработал не идеально — в этом и смысл вопроса. **Retrieval:** BM25 поставил GO-16 (где GPA не обсуждается) выше GO-02 из-за общих слов (`need`, `based`, `grant`). EmbeddingGemma поставила GO-02 на первое место, но добавила в контекст **неверное значение 3.0 на третьем месте**. **Generation:** модель не ошиблась, потому что в GO-05 сказано: "These rules were replaced on 1 June 2026 and do not apply to any application made for the 2026 academic year". Получается, нерелевантный фрагмент появился из-за несовершенного retrieval, но этап generation справился с ним. Опаснее обратная ситуация: в Part B запрос Q-04 ("What is the minimum GPA for the grant?") заставил BM25 вернуть GO-05, GO-17 и GO-19, без **GO-02**. Я не запускал этот запрос через модель, поэтому это предположение. Но если бы так и произошло, значение 2.67 вообще отсутствовало бы в контексте и модель не смогла бы сослаться на него: это была бы ошибка **retrieval**, а не generation.

In [11]:
# Печатает разбор вопроса: результаты поиска, переданный контекст и ответы трёх систем.
def show_case(qid):
# Находим вопрос и результаты трёх вариантов ответа, рассчитанные ранее.
    q = next(x for x in qa if x["id"] == qid); r = runs[qid]
    print(qid, "|", q["question"]); print("REFERENCE:", q["reference"]); print("relevant:", q["relevant"], "\n")
    print("1. BM25 top 3:       ", [(c, round(s, 3)) for c, s in search_bm25(q["question"], 3)])
    print("2. EmbeddingGemma top 3:", [(c, round(s, 3)) for c, s in search_emb(q["question"], 3)])
    for name, key in (("BM25", "bm25_ids"), ("EmbeddingGemma", "emb_ids")):
        print(f"\n3. Context sent to the model by {name} RAG:")
        print(context_block(r[key]))
    print("\nAnswers:")
    for name, key in (("LLM only", "llm"), ("BM25 RAG", "bm25"), ("Embedding RAG", "emb")):
        print(f"- {name}: {r[key]}")

show_case("C-03")

C-03 | What is the minimum GPA needed to qualify for the need-based grant?
REFERENCE: A cumulative GPA of at least 2.67 on the 4.0 scale. (The 3.0 in GO-05 is the archived 2025 rule.)
relevant: ['GO-02'] 

1. BM25 top 3:        [('GO-16', 15.799), ('GO-02', 14.059), ('GO-06', 13.852)]
2. EmbeddingGemma top 3: [('GO-02', 0.629), ('GO-20', 0.569), ('GO-05', 0.545)]

3. Context sent to the model by BM25 RAG:
[1] (GO-16 | Combining the grant with other support)
The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any other source. The total of all university support a student receives in one academic year is capped at 400,000 tenge. When the cap would be exceeded, the need-based grant is reduced so that the total equals the cap.

[2] (GO-02 | Who qualifies)
An applicant qualifies for the 2026 grant when all three conditions hold at the time of the decision: their cumulative GPA is at least 2.67 on the 4.0 scale; their hous

## C3 — question 2 of 3: C-06 (no answer in the corpus)

**1. Что нашёл BM25** (top 3, ID и оценки):

> GO-16 (16.996), GO-01 (14.173), GO-20 (13.72).

**2. Что нашла EmbeddingGemma** (top 3, ID и оценки):

> GO-20 (0.666), GO-16 (0.616), GO-01 (0.554).

**3. Что было передано модели** (блоки контекста):

> BM25: [1] GO-16, [2] GO-01, [3] GO-20 "Who cannot apply". EmbeddingGemma: [1] GO-20, [2] GO-16, [3] GO-01. The key sentence is in GO-20: "Master's and doctoral students are supported by a separate scheme, which this handbook does not describe; they should ask their own department for its rules."

**4. Как retrieval повлиял на ответ:** сделал его точнее, конкретнее и лучше обоснованным или привёл к ошибке из-за нерелевантных chunk?

> **Ответ стал лучше обоснованным — это единственный корректный тип ответа.** Эталонный ответ: "the corpus does not say". В режиме `LLM only` модель этого не сказала: она выдала длинное объяснение о финансировании магистратуры в США (FAFSA, teaching assistantships, "$2,000–$20,000 per year"), то есть ответила на другой вопрос о другой стране. Обе RAG-системы указали, что грант предназначен для студентов бакалавриата очной формы, а магистранты относятся к "a different scheme that isn't detailed here, so the amount you could receive isn't specified in this context". Они сослались на [3] (BM25) и [1] (embeddings); оба указанных блока — GO-20.

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

> Ничего не было упущено. Оба retriever нашли нужный chunk, но с разной надёжностью: EmbeddingGemma поставила GO-20 на первое место (0.666), а BM25 — **на третье** (13.72, ниже GO-16 с 16.996 и GO-01 с 14.173). При K = 2 BM25 потерял бы этот фрагмент. Я не запускал такой вариант, поэтому это риск, а не наблюдавшаяся ошибка. Ответ стал корректным благодаря тому, что **generation следовал system message** ("If the answer cannot be found in the context, say that you do not know"): в контексте нет суммы, и модель не стала её выдумывать. Это видно при сравнении с `LLM only`: та же модель без контекста и такой инструкции уверенно выдала вымышленный ответ. В этом случае retrieval позволил модели сказать "I don't know", но не дал сумму, потому что её нет в источнике.

In [12]:
show_case("C-06")

C-06 | I am a master's student. How much need-based grant can I get?
REFERENCE: The corpus does not say. GO-20 says master's students have a separate scheme that the handbook does not describe.
relevant: ['GO-20'] 

1. BM25 top 3:        [('GO-16', 16.996), ('GO-01', 14.173), ('GO-20', 13.72)]
2. EmbeddingGemma top 3: [('GO-20', 0.666), ('GO-16', 0.616), ('GO-01', 0.554)]

3. Context sent to the model by BM25 RAG:
[1] (GO-16 | Combining the grant with other support)
The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any other source. The total of all university support a student receives in one academic year is capped at 400,000 tenge. When the cap would be exceeded, the need-based grant is reduced so that the total equals the cap.

[2] (GO-01 | About the need-based study grant 2026)
The need-based study grant 2026 is run by the Grant Office for full-time undergraduate students. It exists so that a student from a low

## C3 — question 3 of 3: C-01 (income band 2: amount and payment dates)

**1. Что нашёл BM25** (top 3, ID и оценки):

> GO-04 (20.219), GO-02 (18.742), GO-01 (17.243).

**2. Что нашла EmbeddingGemma** (top 3, ID и оценки):

> GO-04 (0.585), GO-03 (0.484), GO-15 (0.451).

**3. Что было передано модели** (блоки контекста):

> BM25: [1] GO-04 "How much the grant pays" ("A student in band 2 receives 150,000 tenge for the academic year ... first in October and the second in February"), [2] GO-02, [3] GO-01. EmbeddingGemma: [1] GO-04, [2] GO-03 "Income bands", [3] GO-15 "Выплата гранта" (in Russian, payment in October and February, account must be in the student's own name).

**4. Как retrieval повлиял на ответ:** сделал его точнее, конкретнее и лучше обоснованным или привёл к ошибке из-за нерелевантных chunk?

> **Ответ стал точнее и конкретнее.** В режиме `LLM only` модель рассказала об **England maintenance loan** ("£8,500 and £9,500", "three instalments"), потому что в вопросе не указана страна и модель заполнила пробел общими знаниями. С контекстом обе RAG-системы дали ответ **150,000 tenge, two equal instalments, October and February**, совпадающий с `reference`.

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

> В итоговых ответах ничего не упущено, но **retrieval у BM25 был неполным**: в `relevant` указаны GO-04 и GO-15, однако BM25 не вернул GO-15 (русскоязычный фрагмент о выплате не содержит общих токенов с английским вопросом). На второе и третье места попали GO-02 и GO-01, которые не помогают ответить на вопрос. Это не повлияло на результат только потому, что GO-04 уже содержит оба нужных факта: пропущенный chunk не понадобился, то есть пробел в retrieval не привёл к ошибке. Запуск с K = 8 показал небольшой нюанс **generation**: модель написала "75,000 tenge in October and the remaining 75,000 tenge in February". Ни один chunk не указывает 75 000; модель сама вычислила 150 000 / 2. Расчёт верен (в `reference` сказано "two equal instalments"), но он выходит за рамки буквального текста. При K = 8 в контексте также был GO-05 с устаревшим правилом (band 2 получал 120 000), но модель правильно не использовала его. Честный итог: в моём запуске ни один финальный RAG-ответ не оказался неверным, поэтому нельзя указать RAG-ответ хуже `reference`. Ближайшие примеры — проблемы retrieval, которые generation смог компенсировать: BM25 пропустил GO-15 здесь, а в C-03 embeddings добавили устаревший GO-05.

In [13]:
show_case("C-01")

C-01 | How much does a student in income band 2 receive, and when is it paid?
REFERENCE: 150,000 tenge for the academic year, in two equal instalments: October and February.
relevant: ['GO-04', 'GO-15'] 

1. BM25 top 3:        [('GO-04', 20.219), ('GO-02', 18.742), ('GO-01', 17.243)]
2. EmbeddingGemma top 3: [('GO-04', 0.585), ('GO-03', 0.484), ('GO-15', 0.451)]

3. Context sent to the model by BM25 RAG:
[1] (GO-04 | How much the grant pays)
The grant amount depends only on the income band, not on the GPA. A student in band 1 receives 250,000 tenge for the academic year. A student in band 2 receives 150,000 tenge for the academic year. The amount is paid in two equal instalments, the first in October and the second in February. There is no partial grant: a student either receives the full amount for their band or nothing.

[2] (GO-02 | Who qualifies)
An applicant qualifies for the 2026 grant when all three conditions hold at the time of the decision: their cumulative GPA is at least 2.

---
# Final questions / Итоговые вопросы

**1. What is the main difference between BM25 and embedding retrieval?**

> BM25 сравнивает **слова**: считает, сколько токенов запроса встречается в chunk, придавая больший вес редким словам и снижая влияние длинных фрагментов. Embedding retrieval сравнивает **смысл**: модель превращает запрос и каждый chunk в вектор (здесь — 768 чисел), после чего выбираются ближайшие векторы. В A1 (a) BM25 дал всем документам 0, потому что `retrieve` отличается от `retrieves`; embeddings в A2 всё равно поставили D2 на первое место (0.9977), а D1 — на второе (0.9956).

**2. When can BM25 perform better than embeddings?**

> Когда запрос содержит точную редкую строку: код формы, номер комнаты или имя. В Q-09 токены `214` и `room` встречаются каждый только в одном из 74 chunk и дали GO-21 оценку 14.73 против 6.94 у следующего результата. В моём запуске BM25 не превзошёл embeddings по hit@3 (6/10 против 10/10), поэтому его преимущество проявилось в более выраженном ранжировании точных совпадений, а не в большем количестве успешных запросов. Точность может приводить и к ошибкам: в Q-04 совпадение по `minimum` поставило устаревший GO-05 на первое место.

**3. When can embeddings perform better than BM25?**

> Когда запрос и ответ используют разные слова или разные языки. Q-07 ("my mother's card") нашёл GO-15 на русском ("счёт родителей"), хотя у BM25 нет общих токенов. В Q-10 (вопрос на русском, chunk на английском) BM25 получил 0.0000 для всех документов, а EmbeddingGemma поставила GO-10 на первое место (0.5794). В Q-02 запрос ("My family is poor") нашёл chunk с формулировкой "low-income household", тогда как BM25 вывел на первое место фрагмент о Lake Balkhash.

**4. Why is cosine similarity used with embeddings?**

> Cosine similarity измеряет **угол** между двумя векторами и не учитывает их длину. Поэтому сравнивается направление (тема текста), а не количество слов. В A2 вектор D2 = [0.9, 0.3, 0.6] длиннее D1 = [0.7, 0.1, 0.4], но оба почти совпадают по направлению с запросом (0.9977 и 0.9956), тогда как D3 направлен иначе (0.4244). Получается ограниченная шкала оценок, которую удобно сравнивать между chunk. Для нормализованных векторов cosine similarity сводится к скалярному произведению, которое вычисляется быстро.

**5. What information is actually sent to an LLM in a RAG system?**

> Только текст: system instructions, найденные chunk, вставленные как пронумерованные блоки, и вопрос. В C1 это 44 + 279 + 23 = 346 токенов по подсчёту tiktoken (сервер насчитал 420) для трёх chunk общим размером 234 токена. Документы целиком, индекс, векторы и similarity scores не передаются.

**6. Does the LLM search the original documents itself?**

> Нет. Мой код выполнял поиск через BM25 или EmbeddingGemma и вставлял top 3 chunk в prompt; модель только генерирует продолжение этого prompt. Это видно в столбце `LLM only`: без вставленных фрагментов та же модель ответила правильно на 0/6 вопросов и выдумала английский образовательный заём для C-01, GPA 2.5 для C-03 и год основания Narxoz — 1992 — для C-05. Причина в том, что модель не имеет прямого доступа к handbook или corpus.

**7. What happens if retrieval returns an irrelevant chunk?**

> Нерелевантный chunk расходует токены и может ввести модель в заблуждение. Если правильный chunk тоже присутствует, хорошая модель обычно может проигнорировать лишний: в C-03 embeddings добавили в контекст архивный GO-05 (minimum GPA 3.0), но ответ всё равно был 2.67, потому что в chunk сказано, что правило больше не действует. Если правильного chunk нет, ничто не спасёт ответ: в Q-04 top 3 от BM25 (GO-05, GO-17, GO-19) не содержал GO-02, поэтому модель не смогла бы привести значение 2.67.

**8. Why can increasing Top-K sometimes improve RAG and sometimes make it worse?** Report K = 1, 3 and 8 on one question.

> Вопрос C-03 (embeddings): **K = 1** → только GO-02, 249 prompt tokens, ответ 2.67; **K = 3** → GO-02, GO-20, GO-05, 425 токенов, ответ 2.67; **K = 8** → GO-02, GO-20, GO-05, GO-16, GO-01, GO-06, GO-19, GO-17, 920 токенов, ответ 2.67. Ответ не изменился, но вход увеличился в 3.7 раза (249 → 920) и заполнился нерелевантными chunk, включая архивное значение 3.0. Для C-01 K = 1 (GO-04, 244 токена) уже давал правильный ответ; K = 3 добавил GO-15 (457 токенов); K = 8 (964 токена) добавил ещё шесть chunk, после чего модель сама дополнила ответ вычислением (75,000 + 75,000). Большой K полезен, если ответ распределён по нескольким chunk (в Q-01 было 3 релевантных фрагмента) или если лучший chunk не на первом месте (C-06 с BM25: GO-20 был третьим). Но большой K мешает, если дополнительные chunk содержат шум или устаревшие сведения, и в любом случае увеличивает число токенов.

In [14]:
# Проверяем влияние числа найденных фрагментов K (1, 3 и 8) на ответ модели.
# K = 1, 3, 8 with EmbeddingGemma on two questions
# Повторяем сравнение для двух вопросов.
for qid in ("C-03", "C-01"):
    q = next(x for x in qa if x["id"] == qid)
    print("=" * 100); print(qid, "|", q["question"]); print("REFERENCE:", q["reference"])
# Меняем размер контекста и смотрим, как это влияет на ответ и число токенов.
    for K in (1, 3, 8):
        cids = [c for c, _ in search_emb(q["question"], K)]
        a, pt, _ = ask(rag_messages(q["question"], cids))
        print(f"\n--- K = {K}: chunks {cids}, relevant found: {[c for c in cids if c in q['relevant']]}, prompt_tokens {pt}")
        print(a)

C-03 | What is the minimum GPA needed to qualify for the need-based grant?
REFERENCE: A cumulative GPA of at least 2.67 on the 4.0 scale. (The 3.0 in GO-05 is the archived 2025 rule.)

--- K = 1: chunks ['GO-02'], relevant found: ['GO-02'], prompt_tokens 249
The minimum GPA required is **2.67 on the 4.0 scale**【1】.

--- K = 3: chunks ['GO-02', 'GO-20', 'GO-05'], relevant found: ['GO-02'], prompt_tokens 425
The minimum GPA required to qualify for the need‑based grant is **2.67** on the 4.0 scale【1】.

--- K = 8: chunks ['GO-02', 'GO-20', 'GO-05', 'GO-16', 'GO-01', 'GO-06', 'GO-19', 'GO-17'], relevant found: ['GO-02'], prompt_tokens 920
The minimum cumulative GPA required to qualify for the need‑based grant is **2.67 on the 4.0 scale**【1】.
C-01 | How much does a student in income band 2 receive, and when is it paid?
REFERENCE: 150,000 tenge for the academic year, in two equal instalments: October and February.

--- K = 1: chunks ['GO-04'], relevant found: ['GO-04'], prompt_tokens 244
A st

**9. Is a vector database required to build a RAG system?**

> Нет. В notebook векторы хранятся в одном массиве NumPy в памяти — `doc_emb` формы (74, 768), тип `float32`; запрос сравнивается со всеми 74 строками через `model.similarity`. BM25 тоже хранится в памяти как объект `rank_bm25`. Vector database становится полезной при тысячах chunk, когда нужны постоянное хранение, фильтрация и approximate search, но для 74 chunk она не нужна.

**10. Which retrieval approach worked better for this corpus, and what evidence supports that?** Give hit@3 for both over your ten queries and the C2 agreement counts for both RAG systems.

**EmbeddingGemma сработала лучше**: hit@3 составил **10/10** против **6/10** у BM25 (BM25 пропустил Q-02, Q-04, Q-07 и Q-10). В C2 совпадение с `reference` составило **6/6 для BM25 RAG, 6/6 для embedding RAG и 0/6 для LLM only**, поэтому обе RAG-системы показали одинаковый итоговый результат. Разница проявилась на этапе retrieval, но не в этих шести ответах: формулировки вопросов близки к текстам chunk, а K = 3 дал BM25 второй шанс. В корпусе есть английский, русский и казахский языки, перефразирования и архивный фрагмент-ловушка — это преимущества для embeddings. BM25 лучше сохраняет точность при поиске конкретных строк, например `214`.

---
# Conclusion

> Я бы создал **hybrid retriever**: запускал BM25 и EmbeddingGemma вместе, а затем объединял их ранжирования (например, с помощью reciprocal rank fusion). Мои результаты объясняют почему: embeddings находят то, что пропускает BM25 (Q-07, Q-10, Q-02 — другой язык и перефразирования), а BM25 хорошо сохраняет точные строки, которые embeddings могут размывать (`214`, `GR-305`). Для handbook, где встречаются оба типа запросов, недостаточно использовать только один метод. Во-вторых, до поиска я бы обработал **archived chunk**: исключил GO-05 из индекса или пометил его метаданными, поскольку в Q-04 он обошёл правильный фрагмент в BM25 (оценка 12.78, а GO-02 не попал в top 3), а в C-03 всё равно попал в prompt. В-третьих, я бы оставил небольшое значение K (3) и требовал от модели указывать ID chunk: при K = 8 количество токенов выросло в 3.7 раза, но ответ не стал лучше. Наконец, я бы проверил систему на гораздо большем числе вопросов, включая вопросы без ответа, как C-06: шести вопросов, на которых обе RAG-системы набрали 6/6, недостаточно, чтобы сравнить качество их итоговых ответов.